In [2]:
import os
import glob

folder_path = r"D:\darazd\daraz_data"
all_files = glob.glob(folder_path + r"\daraz_reviews_*.csv")

# Check each file size
for file in all_files:
    size = os.path.getsize(file)
    print(file, ":", size, "bytes")

D:\darazd\daraz_data\daraz_reviews_1789502099.csv : 1 bytes
D:\darazd\daraz_data\daraz_reviews_319587682.csv : 5568 bytes
D:\darazd\daraz_data\daraz_reviews_323075036.csv : 5257 bytes
D:\darazd\daraz_data\daraz_reviews_335857201.csv : 3137 bytes
D:\darazd\daraz_data\daraz_reviews_342506493.csv : 9056 bytes
D:\darazd\daraz_data\daraz_reviews_342838000.csv : 8603 bytes
D:\darazd\daraz_data\daraz_reviews_342880345.csv : 3900 bytes
D:\darazd\daraz_data\daraz_reviews_343577781.csv : 1446 bytes
D:\darazd\daraz_data\daraz_reviews_343700080.csv : 7347 bytes
D:\darazd\daraz_data\daraz_reviews_343927190.csv : 11534 bytes
D:\darazd\daraz_data\daraz_reviews_368380161.csv : 8250 bytes
D:\darazd\daraz_data\daraz_reviews_416533626.csv : 9004 bytes
D:\darazd\daraz_data\daraz_reviews_464695300.csv : 9474 bytes
D:\darazd\daraz_data\daraz_reviews_511316421.csv : 13084 bytes
D:\darazd\daraz_data\daraz_reviews_529830248.csv : 3056 bytes
D:\darazd\daraz_data\daraz_reviews_530382305.csv : 5695 bytes
D:\daraz

In [ ]:
import pandas as pd
import os
import glob

folder_path = r"D:\darazd\daraz_data"
all_files = glob.glob(folder_path + r"\daraz_reviews_*.csv")

df_list = []

for file in all_files:
    try:
    
        df = pd.read_csv(file)
        
        if not df.empty:
            df_list.append(df)
        else:
            print("Skipping empty file:", file)
    except pd.errors.EmptyDataError:
        print("Skipping unreadable file:", file)

# Combine all non-empty DataFrames
master_df = pd.concat(df_list, ignore_index=True)

# Drop duplicates
master_df.drop_duplicates(inplace=True)

print("Total reviews in master dataset:", len(master_df))
master_df.head()

Skipping unreadable file: D:\darazd\daraz_data\daraz_reviews_1789502099.csv
Total reviews in master dataset: 2841


,review_text,rating,date,review_length
0,প্যাকেজিক ঠিক করতে হবে।দারাজকে ডেলিভারিতে ফোকা...,4,14 Apr 2025,81
1,অনেক সুন্দর একটি প্রোডাক্ট খুব সীমিত দামে তাইল...,5,15 Sep 2025,72
2,প্যাকেটের রংটা আলাদা হলেও জিনিস ঠিকঠাক ছিল।,5,05 Dec 2025,44
3,ডেলিভারিতে আরও যত্নবান হওয়া দরকার ছিলো।,5,22 Jul 2025,40
4,valo❣️💪😾,5,30 Nov 2023,8


In [2]:
master_df.to_csv(folder_path + r"\Daraz_Master_Reviews2.csv", index=False, encoding="utf-8")
print("Master dataset saved successfully!")

Master dataset saved successfully!


In [3]:
# Review length
master_df['review_length'] = master_df['review_text'].str.len()

# Exclamation count
master_df['exclamation_count'] = master_df['review_text'].str.count('!')

# Capital letter ratio
master_df['capital_ratio'] = master_df['review_text'].str.count(r'[A-Z]') / (master_df['review_length'] + 1)

In [4]:
# Remove null reviews
master_df = master_df.dropna(subset=['review_text'])

# Remove very short reviews (e.g., less than 3 characters)
master_df = master_df[master_df['review_length'] > 3]

# Remove duplicate text
master_df = master_df.drop_duplicates(subset=['review_text'])

print("Clean dataset size:", len(master_df))

Clean dataset size: 1828


In [5]:
# Word count
master_df['word_count'] = master_df['review_text'].str.split().apply(len)

# Average word length
master_df['avg_word_length'] = master_df['review_text'].apply(
    lambda x: sum(len(word) for word in x.split()) / (len(x.split()) + 1)
)

# Question mark count
master_df['question_count'] = master_df['review_text'].str.count(r'\?')

# Digit count
master_df['digit_count'] = master_df['review_text'].str.count(r'\d')

In [6]:
import pandas as pd
import numpy as np
import re

# Remove null reviews
master_df = master_df.dropna(subset=['review_text'])

# Remove duplicates
master_df = master_df.drop_duplicates()

# Lowercase
master_df['clean_text'] = master_df['review_text'].str.lower()

# Remove URLs
master_df['clean_text'] = master_df['clean_text'].str.replace(r'http\S+|www\S+', '', regex=True)

# Remove special characters
master_df['clean_text'] = master_df['clean_text'].str.replace(r'[^a-zA-Z\s]', '', regex=True)

# Remove extra spaces
master_df['clean_text'] = master_df['clean_text'].str.strip()

In [7]:
# Word count
master_df['word_count'] = master_df['clean_text'].str.split().str.len()

# Unique word ratio
master_df['unique_word_ratio'] = (
    master_df['clean_text'].apply(lambda x: len(set(x.split())) / (len(x.split()) + 1))
)

# Average word length
master_df['avg_word_length'] = (
    master_df['clean_text'].apply(lambda x: np.mean([len(w) for w in x.split()]) if len(x.split()) > 0 else 0)
)

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    stop_words='english'
)

X_text = tfidf.fit_transform(master_df['clean_text'])

In [9]:
from scipy.sparse import hstack

numerical_features = master_df[
    ['review_length', 'exclamation_count', 'capital_ratio',
     'word_count', 'unique_word_ratio', 'avg_word_length']
].values

X = hstack([X_text, numerical_features])

In [10]:
from scipy.sparse import hstack

numerical_features = master_df[
    ['review_length', 'exclamation_count', 'capital_ratio',
     'word_count', 'unique_word_ratio', 'avg_word_length']
].values

X = hstack([X_text, numerical_features])

# dataset c

In [ ]:
import pandas as pd
import numpy as np
import re

# Load master dataset
master_df = pd.read_csv("daraz_data/Daraz_Master_Reviews.csv")


master_df = master_df.dropna(subset=['review_text'])


master_df = master_df.drop_duplicates()


def clean_text(text):
    text = text.lower()                      # lowercase
    text = re.sub(r'http\S+|www\S+', '', text)  # remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # remove special chars
    text = re.sub(r'\s+', ' ', text).strip() # remove extra spaces
    return text

master_df['clean_text'] = master_df['review_text'].apply(clean_text)

print("After cleaning:", master_df.shape)
master_df.head()

After cleaning: (513, 5)


,review_text,rating,date,review_length,clean_text
0,জামার সাথে ওরনার কোনো মিল নাই । ওরনাটা ছেরা দি...,5,14 Sep 2025,111,
1,আলহামদুলিল্লাহ যেমন চেয়েছি ঠিক তেমনি পেয়েছি🥰🥰,5,07 Jul 2024,45,
2,অসংখ্য ধন্যবাদ যেভাবে লিখেছেন ঠিক তেমনটা দেয়ার...,5,14 Mar 2024,138,
3,product ta khub vlo chilo asha kori vlo hobe a...,5,06 Jun 2024,165,product ta khub vlo chilo asha kori vlo hobe a...
4,জামা সেলোয়ার ওড়না সব সুতি কাপড় ৫৬০ টাকা হিস...,5,10 Feb 2024,108,


In [14]:
# Review length
master_df['review_length'] = master_df['clean_text'].str.len()

# Word count
master_df['word_count'] = master_df['clean_text'].str.split().str.len()

# Exclamation count
master_df['exclamation_count'] = master_df['review_text'].str.count('!')

# Capital ratio
master_df['capital_ratio'] = (
    master_df['review_text'].str.count(r'[A-Z]') /
    (master_df['review_text'].str.len() + 1)
)

# Remove very short reviews (optional but professional)
master_df = master_df[master_df['word_count'] > 2]

In [15]:
# If rating column exists
master_df['label'] = np.where(master_df['rating'] >= 4, 1, 0)

y = master_df['label']

In [16]:
# If rating column exists
master_df['label'] = np.where(master_df['rating'] >= 4, 1, 0)

y = master_df['label']

In [17]:
from scipy.sparse import hstack

numerical_features = master_df[
    ['review_length', 'word_count',
     'exclamation_count', 'capital_ratio']
].values

X = hstack([X_text, numerical_features])

ValueError: blocks[0,:] has incompatible row dimensions. Got blocks[0,1].shape[0] == 152, expected 1828.

In [18]:
# Remove nulls
master_df = master_df.dropna(subset=['review_text'])

# Remove duplicates
master_df = master_df.drop_duplicates()

# Remove short reviews
master_df = master_df[master_df['word_count'] > 2]

# Reset index (IMPORTANT)
master_df = master_df.reset_index(drop=True)

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    stop_words='english'
)

X_text = tfidf.fit_transform(master_df['clean_text'])

In [20]:
from scipy.sparse import hstack
from scipy.sparse import csr_matrix

numerical_features = master_df[
    ['review_length', 'word_count',
     'exclamation_count', 'capital_ratio']
].values

# Convert to sparse matrix
numerical_sparse = csr_matrix(numerical_features)

# Combine
X = hstack([X_text, numerical_sparse])

In [21]:
print(X_text.shape)
print(numerical_sparse.shape)

(152, 2341)
(152, 4)


In [22]:
from scipy.sparse import hstack
from scipy.sparse import csr_matrix

# Convert numerical to sparse
numerical_sparse = csr_matrix(master_df[
    ['review_length', 'word_count',
     'exclamation_count', 'capital_ratio']
].values)

# Combine
X = hstack([X_text, numerical_sparse])

print("Final Feature Matrix Shape:", X.shape)

Final Feature Matrix Shape: (152, 2345)


In [23]:
import numpy as np

master_df['label'] = np.where(master_df['rating'] >= 4, 1, 0)
y = master_df['label']

In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [25]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [26]:
from sklearn.metrics import classification_report, accuracy_score

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.967741935483871
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.97      1.00      0.98        30

    accuracy                           0.97        31
   macro avg       0.48      0.50      0.49        31
weighted avg       0.94      0.97      0.95        31



d:\Programs\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Programs\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Programs\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
